# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jazaalee/FlyRank-StarterNotebook/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Task type: Scoring / ranking, built on top of a classification model

A content reviewer opens a specific page and decides what to actually do with it. The model itself doesn't touch anything; it just tells a person where to spend their limited review time first, instead of scrolling through hundreds of pages randomly.

Under the hood, this is a classification problem — for each page, I'm predicting the probability that it's declining (my is_declining_label). But that's not the final output people actually use. Nobody wants a flat yes/no on every page — what's useful is a ranked list: sort pages by that probability (plus a few other signals) and hand the reviewer the top candidates first.

So the honest way to describe this lane: the model does classification underneath, but the actual deliverable — the thing a person uses — is a ranking/scoring task. What matters isn't "is this page declining" in isolation, it's "out of everything, which pages deserve a human's attention first, given they can't check them all."

This is really just SEO content-refresh work, When a reviewer opens a flagged page, they're checking things like: is it still getting traffic but hasn't been updated in months? Is it losing ground while demand for it still exists? Is it ranking decently but not getting the clicks it should for that position? If the page really is a problem, they take a content action — rewrite it, expand it, fix the title, or in some cases prune it. If it turns out fine, they just mark it reviewed and move on. The ML part isn't making that editing decision — it's just replacing "where do I even start" guesswork with a defensible, evidence-based shortlist.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
Target or proxy: is_declining_label

What I'd predict is whether a page is "declining" — I'm using is_declining_label, which I built from trend_direction == "down".

It's important to be honest about where this label comes from: it's a proxy, not a true observed outcome. It's a snapshot bucket calculated from the current window of data (basically, "as of right now, is this page's trend flagged as down"), not something that happened after a decision point. A stronger version of this label would be forward-looking — something like "using the last 90 days of features, did the page actually decline over the next 30 days." That would be a real observed outcome instead of a same-window label, but it needs daily time-series data I don't have easy access to yet in the starter dataset.

**For this notebook, I'm sticking with the proxy label** since it's what the starter data supports, but I'm naming it clearly as a proxy — not something I'm treating as ground truth. If I move to the full warehouse data later, upgrading to a real future-window label would make this a stronger, more honest target

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
Success metric: Precision@K

I'm using Precision@K, specifically Precision@20 and Precision@50, since that matches how the output actually gets used. A reviewer isn't going to look at every page in the dataset, they're going to check a fixed number, maybe the top 20 or top 50 the model flags. So the metric that actually matters is: out of those top K pages, how many are genuinely declining?

I'm not using something like plain accuracy because it doesn't reflect the real decision. Accuracy treats every page equally, but most pages in a typical dataset aren't the ones anyone cares about. Precision@K focuses on exactly the pages that would get reviewed, which is the only part of the ranking that actually matters for this use case.

From earlier work, my hand-written rule got a Precision@50 of 0.680, so that's the number any model needs to beat to prove it's actually adding value here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
The unit of analysis

One row in my data is one page, identified by content_id. Each row holds that page's recent search and engagement signals (impressions, how long since it was last updated, its trend direction, average position, CTR) plus the label I'm trying to predict.

This matters because it sets the scope of the whole task: I'm not scoring a client, a query, or a day, I'm scoring individual pages. That's what gets ranked, and that's what a reviewer ends up opening and reviewing one at a time.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jazaalee/FlyRank-StarterNotebook"
REPO_DIR = "FlyRank-StarterNotebook"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Now in:", os.getcwd())

Now in: /content/FlyRank-StarterNotebook


In [2]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

print(df.shape[0], "rows loaded")

30000 rows loaded


In [3]:
# One row = one page (one piece of content), which is the unit of analysis for this lane
lane_slice = df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "trend_direction",
    "is_declining_label",
    "avg_position",
    "ctr",
    "hand_rule_score",
]].copy()

print("One row =", "one page (content_id), with signals describing its recent search performance")
lane_slice.head(10)

One row = one page (content_id), with signals describing its recent search performance


,content_id,client_id,impressions_90d,days_since_last_update,trend_direction,is_declining_label,avg_position,ctr,hand_rule_score
0,content_304f48230142,client_f369cb89fc,3803,20,down,1,10.6,0.76,0
1,content_a1fb4e703a9e,client_4e07408562,15320,25,down,1,20.3,0.05,0
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,down,1,36.5,0.09,0
3,content_331d6c4de07b,client_19581e27de,11751,22,stable,0,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,down,1,44.0,0.13,0
5,content_d4084a4bc775,client_f369cb89fc,3970,20,down,1,8.5,0.03,0
6,content_9a34b442b552,client_8722616204,20,20,down,1,7.0,0.00,0
7,content_a63219c6e95a,client_19581e27de,1724,22,stable,0,21.2,0.06,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,down,1,46.0,0.09,0
9,content_c27558df2b0c,client_19581e27de,1240,104,down,1,4.9,0.16,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
Why ML beats a fixed rule here

My hand-written rule checks two conditions on their own: is the page stale, and is it still visible. The problem is that a fixed rule like that assumes every signal matters the same way for every page, no matter what else is going on with it. But when I actually look at my own decision tree's output, that's not what the data shows.

For example, in my tree, content_age_days splits the pages differently depending on whether impressions_90d was high or low. When impressions were low, age barely mattered. When impressions were high, age became the deciding factor. That means these two signals interact, they only tell you something useful when you look at them together, not separately. A simple if-statement rule like "stale AND visible" can't express that, because it treats staleness and visibility as two independent yes/no checks, not as something that depends on each other.

Writing an if-statement for every possible combination of signals like that isn't realistic to do by hand. That's exactly what a model like a decision tree is good at: finding these conditional, branching relationships in the data on its own, instead of me guessing which combinations matter. From my earlier testing, the hand rule got a Precision@50 of 0.680, while even a simple depth-3 tree reached 0.720, and the original starter pipeline's random forest reached 0.740. That gap is the real evidence that these interactions matter enough to be worth catching.

That said, this isn't an argument for a black-box model either. Keeping the tree shallow, or checking feature importances from a random forest, still lets me explain why a page got flagged, which matters because a human still has to trust and act on the recommendation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.